In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [9]:
def get_html(url = 'https://sindipetroce-pi.org.br/comunicacao/page/1/'):
    payload = {}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
        "Referer": "https://google.com"
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [29]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')
    dates = soup.find_all('p', class_='fusion-single-line-meta')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for i in range(len(articles)):
        a = articles[i].find('a')
        link = a.get('href')
        span = dates[i].find('span')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%d de %B de %Y")
                
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue


    return news_links

In [35]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [12]:
def get_next_page(fnp_url = 'https://sindipetroce-pi.org.br/comunicacao/page/', next_page_number = 1):
    validated_news_links = []
    url = fnp_url + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [14]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                              .replace('\xa0',' ')\
                                                              .replace('\n',' ')\
                                                              .replace('\t',' ')\
                                                              .replace('[email-protected]', '')\
                                                              .strip() \
                                                              for paragraph in paragraphs] \
                  if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0
    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [37]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find('div', class_='post-content')

    paragraphs = paragraphs.text.split('\n')
    paragraphs = sanitize_paragraphs(paragraphs)

    return title, paragraphs

In [40]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://sindipetroce-pi.org.br/comunicacao/page/'
    url = url_default + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'RS',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [41]:
result = main()
print(len(result))
result

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.40s/it]

19


[{'sindicato': 'RS',
  'url': 'https://sindipetroce-pi.org.br/2025/08/primeira-mesa-tematica-da-12a-plenafup-debate-conjuntura-nacional/',
  'titulo': 'Primeira mesa temática da 12ª Plenafup debate Conjuntura Nacional',
  'data': datetime.datetime(2025, 8, 6, 0, 0),
  'paragrafo': 'Rosa Amorim, Sônia Meire e Divanilton Pereira discutem os desafios do Brasil e da classe trabalhadora na mesa que marcou o primeiro dia da plenária [Por Vítor Peruch e Guilherme Weimann, da comunicação do Sindipetro Unificado] A 12ª Plenafup iniciou nesta segunda-feira (4) com a realização da Mesa 1: Conjuntura Nacional, que reuniu lideranças políticas e sindicais para uma análise crítica do cenário brasileiro e dos impactos do atual momento político sobre a classe trabalhadora. A mesa teve início após a eleição da mesa diretora da plenária, a aprovação do regimento interno e a apresentação das teses das correntes políticas presentes no encontro. Com mediação do coordenador-geral da FUP, Deyvid Bacelar, e do